# 🚍 Sistema de Transporte Masivo: Agrupamiento No Supervisado con K-Means
### **Actividad de Inteligencia Artificial - Segmentación Operativa y Toma de Decisiones**

---

### **Objetivo del Proyecto:**
Segmentar y perfilar estaciones/franjas horarias de un Sistema de Transporte Masivo (ej. Metro, TransMilenio, MIO) mediante técnicas de **Machine Learning No Supervisado (K-Means)**. El propósito es optimizar la asignación de flota, gestionar tiempos de espera y reducir congestión en puntos críticos.

### **Estructura del Proyecto:**
1. **Simulación del Dataset (1,000 registros)** y guardado en CSV.
2. **Procesamiento y Análisis Exploratorio de Datos (EDA)** con escalado `StandardScaler`.
3. **Pruebas y Validación del Hiperparámetro K** (Método del Codo y Coeficiente de Silueta).
4. **Entrenamiento del Modelo K-Means** y asignación de etiquetas.
5. **Visualización e Interpretación de Negocio** para la sustentación y toma de decisiones operativas.

## 0. Importación de Librerías y Configuración del Entorno

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn para Preprocesamiento y Clustering
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

# Configuración de visualizaciones
%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
print("Librerías importadas exitosamente.")

## 1. Simulación y Generación del Dataset (`datos_transporte_masivo.csv`)
> **Explicación para la sustentación:** Se simulan 1,000 observaciones que representan el comportamiento del sistema en tres realidades operativas: **Horas Valle**, **Horas Intermedias** y **Horas Pico / Congestión Crítica**.

In [ ]:
def generar_dataset_transporte(n_samples=1000, random_state=42):
    np.random.seed(random_state)
    
    n_valle = int(n_samples * 0.35)
    n_moderado = int(n_samples * 0.40)
    n_pico = n_samples - n_valle - n_moderado
    
    # 1. Valle / Baja Demanda
    afluencia_valle = np.random.normal(loc=180, scale=40, size=n_valle).clip(50, 320)
    espera_valle = np.random.normal(loc=3.5, scale=0.8, size=n_valle).clip(1, 6)
    retraso_valle = np.random.normal(loc=1.2, scale=0.4, size=n_valle).clip(0.2, 3)
    recargas_valle = np.random.normal(loc=90, scale=25, size=n_valle).clip(20, 180)
    franja_valle = np.random.choice([1], size=n_valle)

    # 2. Moderado / Flujo Regular
    afluencia_mod = np.random.normal(loc=700, scale=90, size=n_moderado).clip(450, 950)
    espera_mod = np.random.normal(loc=8.5, scale=1.2, size=n_moderado).clip(5.5, 12)
    retraso_mod = np.random.normal(loc=4.5, scale=0.9, size=n_moderado).clip(2, 7)
    recargas_mod = np.random.normal(loc=380, scale=60, size=n_moderado).clip(220, 550)
    franja_mod = np.random.choice([2], size=n_moderado)

    # 3. Crítico / Hora Pico
    afluencia_pico = np.random.normal(loc=1650, scale=180, size=n_pico).clip(1200, 2400)
    espera_pico = np.random.normal(loc=19.0, scale=2.5, size=n_pico).clip(13, 28)
    retraso_pico = np.random.normal(loc=13.5, scale=2.0, size=n_pico).clip(8, 22)
    recargas_pico = np.random.normal(loc=980, scale=130, size=n_pico).clip(650, 1500)
    franja_pico = np.random.choice([3], size=n_pico)

    df = pd.DataFrame({
        'id_registro': np.arange(1, n_samples + 1),
        'afluencia_pasajeros': np.round(np.concatenate([afluencia_valle, afluencia_mod, afluencia_pico]), 0).astype(int),
        'tiempo_espera_promedio': np.round(np.concatenate([espera_valle, espera_mod, espera_pico]), 2),
        'retraso_promedio_bus': np.round(np.concatenate([retraso_valle, retraso_mod, retraso_pico]), 2),
        'tarjetas_recargadas': np.round(np.concatenate([recargas_valle, recargas_mod, recargas_pico]), 0).astype(int),
        'franja_horaria_cod': np.concatenate([franja_valle, franja_mod, franja_pico])
    })
    
    df = df.sample(frac=1, random_state=random_state).reset_index(drop=True)
    df['id_registro'] = np.arange(1, n_samples + 1)
    return df

# Guardar en CSV
nombre_csv = 'datos_transporte_masivo.csv'
df_transporte = generar_dataset_transporte(n_samples=1000, random_state=42)
df_transporte.to_csv(nombre_csv, index=False, encoding='utf-8')
print(f"Dataset generado con éxito ({len(df_transporte)} filas) y guardado en '{nombre_csv}'.")
df_transporte.head()

## 2. Procesamiento, EDA y Escalado de Variables
> **¿Por qué escalar con StandardScaler?**
> K-Means utiliza distancias euclidianas. Si no escalamos, variables con magnitudes de miles (`afluencia_pasajeros`) sesgarían el cálculo frente a variables de menor magnitud (`tiempo_espera_promedio` o `retraso_promedio_bus`). StandardScaler transforma cada columna para que tenga media 0 y varianza 1.

In [ ]:
# Carga y verificación de nulos
df = pd.read_csv(nombre_csv)
print("Nulos por columna:")
print(df.isnull().sum())

# Selección de características predictoras
features_clustering = [
    'afluencia_pasajeros', 
    'tiempo_espera_promedio', 
    'retraso_promedio_bus', 
    'tarjetas_recargadas', 
    'franja_horaria_cod'
]
X = df[features_clustering].copy()

# Escalado estándar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\nMatriz escalada lista. Primeras 3 filas:")
print(pd.DataFrame(X_scaled, columns=features_clustering).head(3))

## 3. Pruebas y Selección del Hiperparámetro K: Codo y Silueta
- **Método del Codo (Inercia):** Evalúa la reducción de la suma de distancias al cuadrado intra-cluster.
- **Coeficiente de Silueta:** Mide la cohesión interna frente a la separación con otros clusters (rango de -1 a 1, donde valores cercanos a 1 indican una separación excelente).

In [ ]:
# Cálculo del Método del Codo (Inercia)
inercia = []
rango_k = range(1, 11)
for k in rango_k:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_scaled)
    inercia.append(km.inertia_)

# Cálculo de Coeficiente de Silueta para K de 2 a 6
silhouette_scores = {}
for k in range(2, 7):
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores[k] = score
    print(f"K = {k} -> Silhouette Score: {score:.4f}")

# Gráficas comparativas
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Gráfica del Codo
axes[0].plot(rango_k, inercia, marker='o', color='#1f77b4', linewidth=2.5, markersize=8)
axes[0].set_title('Método del Codo (Inercia vs K)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Número de Clusters (K)')
axes[0].set_ylabel('Inercia')
axes[0].set_xticks(rango_k)
axes[0].grid(True, linestyle='--', alpha=0.7)

# 2. Gráfica de Silueta
k_vals = list(silhouette_scores.keys())
s_vals = list(silhouette_scores.values())
bars = axes[1].bar(k_vals, s_vals, color='#2ca02c', edgecolor='black', alpha=0.85)
axes[1].set_title('Coeficiente de Silueta Promedio por K', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Número de Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_ylim(0, 1.0)
axes[1].set_xticks(k_vals)
axes[1].grid(True, linestyle='--', alpha=0.7)

for bar in bars:
    height = bar.get_height()
    axes[1].annotate(f'{height:.3f}',
                     xy=(bar.get_x() + bar.get_width() / 2, height),
                     xytext=(0, 4), textcoords="offset points",
                     ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

k_optimo = max(silhouette_scores, key=silhouette_scores.get)
print(f"\n>> JUSTIFICACIÓN TÉCNICA: Se selecciona K = {k_optimo} debido a que presenta el mayor coeficiente de silueta ({silhouette_scores[k_optimo]:.4f}) y define el punto de inflexión del codo.")

## 4. Entrenamiento y Asignación de Clusters Finales

In [ ]:
# Configuración y ajuste del modelo con el K óptimo
kmeans_final = KMeans(n_clusters=k_optimo, init='k-means++', n_init=20, max_iter=300, random_state=42)
df['cluster'] = kmeans_final.fit_predict(X_scaled)

print("Distribución de registros por cluster:")
print(df['cluster'].value_counts().sort_index())

## 5. Visualización e Interpretación para la Toma de Decisiones

In [ ]:
# Obtener centroides en la escala original
centroides_originales = scaler.inverse_transform(kmeans_final.cluster_centers_)
df_centroides = pd.DataFrame(centroides_originales, columns=features_clustering)
df_centroides['cluster'] = range(k_optimo)

# Gráfico Scatter Plot 2D
plt.figure(figsize=(12, 7))
colores = ['#2ca02c', '#d62728', '#1f77b4']

sns.scatterplot(
    data=df,
    x='afluencia_pasajeros',
    y='tiempo_espera_promedio',
    hue='cluster',
    palette=colores,
    alpha=0.65,
    s=55,
    edgecolor='none'
)

# Dibujar Centroides
plt.scatter(
    df_centroides['afluencia_pasajeros'],
    df_centroides['tiempo_espera_promedio'],
    color='black',
    marker='X',
    s=250,
    edgecolor='yellow',
    linewidth=1.8,
    label='Centroides de Cluster'
)

for i, row in df_centroides.iterrows():
    plt.annotate(
        f'Centroide {int(row["cluster"])}',
        (row['afluencia_pasajeros'], row['tiempo_espera_promedio']),
        textcoords="offset points", xytext=(0, 14), ha='center',
        fontsize=11, fontweight='bold',
        bbox=dict(boxstyle="round,pad=0.3", fc="yellow", alpha=0.8, ec="black")
    )

plt.title('Segmentación Operativa del Sistema de Transporte con K-Means', fontsize=14, fontweight='bold')
plt.xlabel('Afluencia de Pasajeros (personas/hora)', fontsize=12)
plt.ylabel('Tiempo de Espera Promedio (minutos)', fontsize=12)
plt.legend(title='Grupo / Cluster', loc='upper left')
plt.tight_layout()
plt.show()

### **Perfilamiento Estadístico y Caracterización de los Segmentos**

In [ ]:
resumen_clusters = df.groupby('cluster')[features_clustering].mean().round(2)
resumen_clusters['total_registros'] = df['cluster'].value_counts()
resumen_clusters['porcentaje'] = (df['cluster'].value_counts(normalize=True) * 100).round(1).astype(str) + '%'
print("RESUMEN ESTADÍSTICO POR CLUSTER:")
display(resumen_clusters)

# Asignación de nombres de perfil operativo
def etiquetar_cluster(row):
    if row['afluencia_pasajeros'] > 1200:
        return 'Estaciones Críticas / Alta Demanda (Pico)'
    elif row['afluencia_pasajeros'] > 500:
        return 'Operación Moderada / Flujo Regular'
    else:
        return 'Operación Valle / Baja Demanda'

df['perfil_operativo'] = df.apply(etiquetar_cluster, axis=1)
df.to_csv('datos_transporte_masivo_clusterizado.csv', index=False, encoding='utf-8')
print("Dataset final etiquetado guardado como 'datos_transporte_masivo_clusterizado.csv'.")

## 6. Guía de Interpretación para la Sustentación en Video

| Cluster | Perfil Operativo | Características Promedio | Decisiones Estratégicas y Operativas |
| :--- | :--- | :--- | :--- |
| **Cluster 1 (Rojo)** | **Estaciones Críticas / Hora Pico** | Afluencia: ~1,635 pax/h<br>Espera: ~19 min<br>Retraso bus: ~13.5 min | **Inyección de buses expresos de refuerzo**, habilitación de taquillas adicionales y recarga digital, gestores de flujo en andenes para evitar saturación. |
| **Cluster 2 (Azul)** | **Operación Moderada / Flujo Regular** | Afluencia: ~708 pax/h<br>Espera: ~8.5 min<br>Retraso bus: ~4.5 min | **Mantenimiento de frecuencias estándar**, control de regularidad en intervalos y sincronización semafórica de carriles exclusivos. |
| **Cluster 0 (Verde)** | **Operación Valle / Baja Demanda** | Afluencia: ~180 pax/h<br>Espera: ~3.5 min<br>Retraso bus: ~1.2 min | **Reducción de flota activa** para optimizar costos de combustible y desgaste de vehículos; redistribución de personal hacia zonas críticas. |